In [0]:
-- Paso 1: Crear vista temporal desde el archivo CSV
-- Usa ruta fija que siempre se sobrescribe (permite automatización)
CREATE OR REPLACE TEMPORARY VIEW temp_bronze_raw AS
SELECT *
FROM read_files(
  '/Volumes/prueba_api/landing/archivos/data_engineer_jobs_latest.csv/*.csv',
  format => 'csv',
  header => true,
  escape => '"',
  quote => '"'
);

In [0]:
-- Paso 2: MERGE a tabla Bronze (preservando registros antiguos, evitando duplicados)
-- Deduplicar el source antes del MERGE (el CSV puede contener job_id duplicados)
MERGE INTO prueba_api.bronze.jobs_bronze AS target
USING (
  SELECT *
  FROM (
    SELECT *, 
           ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY job_posted_at_timestamp DESC) as rn
    FROM temp_bronze_raw
  )
  WHERE rn = 1
) AS source
ON target.job_id = source.job_id

-- Si el job_id ya existe, actualizar los datos
WHEN MATCHED THEN
    UPDATE SET *

-- Si el job_id no existe, insertar el nuevo registro
WHEN NOT MATCHED THEN
    INSERT *;

In [0]:
-- Verificar que la tabla Bronze se creó correctamente
SELECT 
    COUNT(*) as total_registros,
    COUNT(DISTINCT job_id) as registros_unicos
FROM prueba_api.bronze.jobs_bronze;